[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/byu-matrix-lab/torchlingo/blob/main/docs/docs/course/lecture-11-comet-on-a6.ipynb)


# Lecture 11: COMET on your A6 sentences, against BLEU, chrF and your own ranking

In A6 you scored two MT systems with BLEU and chrF and ranked 50 of their translations by hand.
This adds two learned metrics, COMET with a reference and COMET-QE-DA without one, and asks the
question A11 is about: do the metrics agree with each other, and with you?

By the end you will be able to:

1. score both systems with COMET and COMET-QE-DA on the same sentences as BLEU and chrF;
2. measure how often two metrics prefer the same system, sentence by sentence;
3. say whether each COMET model agrees with BLEU, chrF and your ranking, and why.

You write two functions, marked TODO; checks tell you when each is right, and the notebook runs
the rest. **Running in Google Colab?** Runtime → Change runtime type → any GPU (a T4 is fine),
then Run all; it stops at each TODO until you write it. When you finish, disconnect the GPU: Runtime → Disconnect and delete runtime. The last cell says why.


<!-- GENERATED:purpose - change the torchlingo metadata, not this cell -->

**CS 479 course notebook** — homework for Lecture 11 (*Neural Quality Estimation and Evaluation*). A head start on **A11**.

In [ ]:
# Install TorchLingo and COMET in Colab. Nothing to uncomment. Pip's "you may need to restart
# the kernel" does not apply to a first install; if the install fails, the import below stops here.
import sys
if "google.colab" in sys.modules:
    %pip install --quiet "torchlingo @ git+https://github.com/byu-matrix-lab/torchlingo@course"
    # pip keeps a TorchLingo already in this session when its version number matches, even if
    # it is older code; this swaps in the course branch's, leaving the dependencies alone.
    %pip install --quiet --force-reinstall --no-deps "torchlingo @ git+https://github.com/byu-matrix-lab/torchlingo@course"
    %pip install --quiet unbabel-comet
import torchlingo


In [ ]:
# Check the setup and connect Google Drive, where your A6 files are.
from torchlingo.colab import setup

env = setup(drive=True)
A6_DIR = env.drive_dir / "CS479" / "assignment6"


## Step 1: your A6 sentences

Put your A6 files in `MyDrive/CS479/assignment6`, one sentence per line, line *n* of each
belonging to source sentence *n*: the sources, each system's translations, and the references.
Set the file names below, name the two systems, and type the two MTEval average ranks you
reported in A6 (closer to 1 is better).


In [ ]:
SOURCE_FILE = "source.txt"          # <- your file names, in MyDrive/CS479/assignment6
REFERENCE_FILE = "reference.txt"
SYSTEM_FILES = {"system 1": "system1.txt", "system 2": "system2.txt"}   # <- e.g. "DeepL": ...
HUMAN_AVERAGE_RANK = {"system 1": None, "system 2": None}           # <- from A6, e.g. 1.42


def read_lines(name):
    path = A6_DIR / name
    if not path.exists():
        raise SystemExit(f"Not found: {path}. Put your A6 files there, or fix the name above.")
    return path.read_text(encoding="utf-8").splitlines()


sources, references = read_lines(SOURCE_FILE), read_lines(REFERENCE_FILE)
translations = {name: read_lines(f) for name, f in SYSTEM_FILES.items()}
assert set(HUMAN_AVERAGE_RANK) == set(SYSTEM_FILES), "name the same two systems in both places"
print(f"{len(sources):,} source sentences; systems: {', '.join(SYSTEM_FILES)}.")


## Step 2: `comet_inputs`, from A10

**Yours: the function you wrote in A10**, which turns aligned sentences into the records COMET
scores and refuses lists of different lengths. Paste it in place of the TODO; the same checks
run.


In [ ]:
def comet_inputs(sources, translations, references=None):
    """Return one COMET record per sentence: src, mt, and ref when references are given.

    TODO: paste your A10 function. It raises ValueError if translations, or references when
    given, are not the same length as sources.
    """
    raise NotImplementedError("paste comet_inputs from A10, then run this cell again")


assert comet_inputs(["hola"], ["hello"], ["hi"]) == [{"src": "hola", "mt": "hello", "ref": "hi"}], (
    "with a reference, each record is {'src': ..., 'mt': ..., 'ref': ...}")
assert comet_inputs(["hola"], ["hello"]) == [{"src": "hola", "mt": "hello"}], (
    "without a reference, each record has only 'src' and 'mt'")
for bad in ((["a", "b"], ["x"], None), (["a"], ["x"], ["r", "s"])):
    try:
        comet_inputs(*bad)
    except ValueError:
        continue
    raise AssertionError(f"comet_inputs must refuse lists of different lengths: {bad}")
for name, mt in translations.items():
    comet_inputs(sources, mt, references)   # your files are aligned, or this says which is not
print("comet_inputs passes its checks, and your files are aligned.")


## Step 3: yours to write, `pairwise_agreement`

With two systems, "do two metrics correlate?" has a sentence-by-sentence answer: for each
sentence, does each metric prefer the same system? `pairwise_agreement` takes two metrics'
scores for both systems and returns how many sentences they agree on, out of how many both
express a preference on. Higher is better for every score here; a tie is no preference.


In [ ]:
def pairwise_agreement(a_1, a_2, b_1, b_2):
    """Return (agree, counted) for metric a's and metric b's scores of systems 1 and 2.

    TODO: write it. For sentence i, metric a prefers system 1 if a_1[i] > a_2[i], system 2 if
    a_1[i] < a_2[i], and neither if they are equal; likewise b. Count only the sentences where
    both metrics prefer a system, and of those, how many prefer the same one.
    """
    raise NotImplementedError("write pairwise_agreement, then run this cell again")


# A case you can check by hand: agree, disagree, a tie in a, a tie in b, agree.
assert pairwise_agreement([2, 1, 5, 3, 0.9], [1, 2, 5, 1, 0.1],
                          [9, 9, 1, 4, 0.7], [1, 1, 2, 4, 0.2]) == (2, 3), (
    "pairwise_agreement is not right yet: here it should count 2 agreements out of 3")
print("pairwise_agreement passes its check.")


## Step 4: predict, then score

**Write a prediction** in `PREDICTION`: will COMET prefer the same system as BLEU, chrF and your
ranking, and will COMET-QE-DA agree with COMET? Then the cell scores both systems with all four
metrics, on every sentence. COMET-QE-DA needs no Hugging Face token. On a GPU this takes a few
minutes for 500 sentences.


In [ ]:
PREDICTION = ""
if not PREDICTION.strip():
    print("No prediction written. Write one before scoring; the report will say it is missing.\n")

import torch
from comet import download_model, load_from_checkpoint
from sacrebleu.metrics import CHRF
from torchlingo.evaluation import compute_bleu, compute_chrf

GPUS = 1 if torch.cuda.is_available() else 0
sentence_chrf = CHRF(word_order=2)          # the chrF++ that compute_chrf uses

system_scores, sentence_scores = {}, {}
for model_name, label, with_refs in (("Unbabel/wmt22-comet-da", "COMET", True),
                                     ("Unbabel/wmt20-comet-qe-da", "COMET-QE-DA", False)):
    model = load_from_checkpoint(download_model(model_name))
    for name, mt in translations.items():
        out = model.predict(comet_inputs(sources, mt, references if with_refs else None),
                            batch_size=32, gpus=GPUS, num_workers=2, progress_bar=False)
        system_scores[(label, name)] = out.system_score
        sentence_scores[(label, name)] = out.scores

for name, mt in translations.items():
    system_scores[("BLEU", name)] = compute_bleu(mt, references).score
    system_scores[("chrF", name)] = compute_chrf(mt, references).score
    sentence_scores[("chrF", name)] = [sentence_chrf.sentence_score(h, [r]).score
                                       for h, r in zip(mt, references)]
print("Scored with COMET, COMET-QE-DA, BLEU and chrF.")


## Step 5: the report

The system scores, which system each metric prefers, and, sentence by sentence, how often the
metrics agree. With two systems, "correlates with your ranking" at the system level means
"prefers the same system"; the sentence-level counts say how firmly.


In [ ]:
one, two = SYSTEM_FILES
print("=== For the A11 summary ===")
print(f"{len(sources):,} sentences scored automatically; systems: {one} and {two}\n")
print(f"{'metric':12} {one:>12} {two:>12}   prefers")
for label in ("BLEU", "chrF", "COMET", "COMET-QE-DA"):
    s1, s2 = system_scores[(label, one)], system_scores[(label, two)]
    print(f"{label:12} {s1:12.3f} {s2:12.3f}   {one if s1 > s2 else two}")
h1, h2 = HUMAN_AVERAGE_RANK[one], HUMAN_AVERAGE_RANK[two]
if h1 is None or h2 is None:
    print(f"{'human rank':12} {'(type your A6 average ranks in Step 1)':>27}")
else:
    print(f"{'human rank':12} {h1:12.2f} {h2:12.2f}   {one if h1 < h2 else two}  (lower is better)")

print("\nSentence by sentence, how often two metrics prefer the same system:")
for a, b in (("COMET", "chrF"), ("COMET-QE-DA", "chrF"), ("COMET-QE-DA", "COMET")):
    agree, counted = pairwise_agreement(sentence_scores[(a, one)], sentence_scores[(a, two)],
                                        sentence_scores[(b, one)], sentence_scores[(b, two)])
    print(f"  {a} and {b}: {agree} of {counted} ({100 * agree / max(counted, 1):.0f}%)")
print(f"\nYour prediction: {PREDICTION.strip() or '(none written)'}")


## Step 6: the summary

A11's one-page summary reports the numbers above, and answers: does each COMET model agree with
BLEU, chrF and your ranking, and why or why not? Check the reference-based COMET and COMET-QE-DA
separately, since their scores sit on different scales. Reconcile your prediction with the
result in a sentence. For the extra credit, xCOMET-XL is `Unbabel/XCOMET-XL` in the same loop;
it needs a Hugging Face token and a large GPU.


---
## When you finish: disconnect the GPU

**Runtime → Disconnect and delete runtime.** Colab charges compute units for every minute a GPU runtime stays connected, even when nothing is running, so a runtime left open after the work is done uses up your monthly units. Closing the browser tab does not end it right away. Copy the numbers your A11 summary needs first; your A6 files stay on Drive.